<a href="https://colab.research.google.com/github/jash2603/leaf-species-classifier/blob/main/leaf_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf4
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Define paths
DATASET_PATH = "/content/drive/MyDrive/Leaf"# Set to actual dataset path

# Define hyperparameters
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 60

# ImageDataGenerator for Train & Validation (70-15 split)
train_val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.3,  # 30% of data used for validation+test
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # For multi-class classification
    subset='training'
)

validation_generator = train_val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# Get the number of classes directly from the generator
num_classes = train_generator.num_classes

# Build CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')  # Uses num_classes instead of 5
])


# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS
)

# Evaluate on Test Set
# test_loss, test_acc = model.evaluate(test_generator)
# print(f"\nTest Accuracy: {test_acc:.4f}")

# Plot Accuracy & Loss
def plot_metrics(history):
    plt.figure(figsize=(12, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Training vs Validation Accuracy')

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training vs Validation Loss')

    plt.show()

plot_metrics(history)



In [ ]:
model.save('/content/drive/MyDrive/Leaf/leaf_model.h5')

https://accounts.google.com/SignOutOptions?hl=en&continue=https://colab.research.google.com/drive/1zTs5O9rHHgmZJFPHeS-CLwgc7ggNTE_c&ec=GBRAqQM$0

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

In [ ]:
# This step is only applicable if model is already trained and that model is saved in drive.
import tensorflow as tf
model = tf.keras.models.load_model('/content/drive/MyDrive/Leaf/leaf_model.h5')
model.summary()

In [ ]:
img_path = "/content/drive/MyDrive/Leaf/Rosa-Senensis/img_0006.jpg"  # Replace with your image path


from tensorflow.keras.preprocessing import image
import numpy as np

IMG_SIZE = (128, 128)  # Same as training size

img = image.load_img(img_path, target_size=IMG_SIZE)
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # Make batch of 1





In [ ]:
# class_labels = ['Tulsi', 'Rosa-Senensis', 'money', 'Mango', 'Lemon']  # Replace with your actual class folder names


# predicted_class = class_labels[np.argmax(pred)]
# print(f"Predicted Plant: {predicted_class}")

# Make prediction
pred = model.predict(img_array, verbose=0)   # shape: (1, 5)
pred_idx = int(np.argmax(pred[0]))

# Map index -> class name using the generator (recommended)
idx_to_class = {v: k for k, v in train_generator.class_indices.items()}
predicted_class = idx_to_class[pred_idx]

print(f"Predicted Plant: {predicted_class}  (prob={pred[0][pred_idx]:.3f})")

# Optional: if you really want to use your own list (must be same order as generator):
# class_labels = ['Tulsi', 'Rosa-Senensis', 'money', 'Mango', 'Lemon']
# predicted_class = class_labels[pred_idx]


In [ ]:
import matplotlib.pyplot as plt

plt.imshow(img)
plt.title(f"Predicted: {predicted_class}")
plt.axis('off')
plt.show()